# Get the alias of each installation's root collection

# Import libraries and set variables
Imports are all in this cell. Re-uses `get_root_collection_info` from `dataverse_functions`.

In [1]:
import os
import pandas as pd
import platform
import sys
from time import sleep
from tqdm import tqdm
tqdmBarFormat = "{l_bar}{bar:10}| {n_fmt}/{total_fmt} [Elapsed:{elapsed}<Remaining:{remaining}{rate_fmt}{postfix}]"

osName = platform.system()
if osName == 'Windows':
    dataverseFunctionsPath = os.path.join(
        'C:\\', 'Users', 'Owner', 'Documents', 'GitHub', 'dataverse-functions')
elif osName == 'Darwin':
    dataverseFunctionsPath = os.path.join(
        '/Users', 'juliangautier', 'dataverse-functions')
sys.path.append(dataverseFunctionsPath)
from dataverse_functions import dataverse_functions

# Some Dataverse installations block requests from scripts, so send a user agent and email address
userAgent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/81.0.4044.138 Safari/537.36'
emailAddress = 'juliangautier@g.harvard.edu'
headers = {
    'User-Agent': userAgent,
    'From': emailAddress}

processedDataDir = '/Users/juliangautier/ux-research-data-analysis/Findability_of_author_affiliations_with_rors/data/processed'
summaryCountsFilePath = os.path.join(processedDataDir, 'installations', 'summary_counts.csv')
outputFilePath = os.path.join(processedDataDir, 'root_collection_aliases.csv')

# Load the installations that have author affiliation RORs
Installations with 0 distinct affiliation names (QDR_Main_Collection and Libra_Data) aren't in the report, so they're dropped.

In [2]:
installationsDf = pd.read_csv(summaryCountsFilePath, encoding='utf-8-sig')

# Keep only installations that have author affiliations
installationsDf = installationsDf[installationsDf['distinct_affiliation_name_count'] > 0].reset_index(drop=True)
installationsDf.head()

,installation_name,installation_url,distinct_affiliation_name_count
0,Harvard_Dataverse,https://dataverse.harvard.edu,374
1,DataverseNL,https://dataverse.nl,122
2,Texas_Data_Repository_Dataverse,https://dataverse.tdl.org,78
3,DataverseNO,https://dataverse.no,51
4,Dataverse_e_cienciaDatos,https://edatos.consorciomadrono.es,40


# Data quality checks
Check for missing values, duplicate installation URLs and a row count of 14

In [3]:
print(f'Row count (expecting 14): {len(installationsDf)}')
print('Missing values per column:')
print(installationsDf.isna().sum())
print(f"Duplicate installation URLs: {installationsDf['installation_url'].duplicated().sum()}")

Row count (expecting 14): 14
Missing values per column:
installation_name                  0
installation_url                   0
distinct_affiliation_name_count    0
dtype: int64
Duplicate installation URLs: 0


# Get the root collection alias of each installation
Uses `get_root_collection_info`, which calls each installation's `/api/dataverses/:root` endpoint. Sleeps between calls and keeps going if an installation fails.

In [4]:
rootAliasesList = []
failuresList = []

loopObj = tqdm(bar_format=tqdmBarFormat, iterable=installationsDf.to_dict('records'))
for row in loopObj:
    installationUrl = row['installation_url']
    loopObj.set_postfix_str(installationUrl)

    rootAlias = None
    rootDatabaseId = None
    try:
        rootCollectionInfoDict = dataverse_functions.get_root_collection_info(
            installationUrl, headers=headers)
        rootAlias = rootCollectionInfoDict['rootAlias']
        rootDatabaseId = rootCollectionInfoDict['rootDatabaseId']
    except Exception as e:
        failuresList.append({
            'installation_url': installationUrl,
            'error': f'{type(e).__name__}: {e}'})

    rootAliasesList.append({
        'installation_name': row['installation_name'],
        'installation_url': installationUrl,
        'root_database_id': rootDatabaseId,
        'root_alias': rootAlias})

    # Pause between calls to avoid rate limits
    sleep(2)

rootAliasesDf = pd.DataFrame(rootAliasesList)
print(f'Failures: {len(failuresList)}')
for failure in failuresList:
    print(failure)
rootAliasesDf.head(14)

100%|██████████| 14/14 [Elapsed:00:38<Remaining:00:00 2.75s/it, https://dataverse.csuc.cat]               

Failures: 3
{'installation_url': 'https://dataverse.tdl.org', 'error': 'JSONDecodeError: Expecting value: line 1 column 1 (char 0)'}
{'installation_url': 'https://dataverse.whoi.edu', 'error': 'JSONDecodeError: Expecting value: line 1 column 1 (char 0)'}
{'installation_url': 'https://dataverse.csuc.cat', 'error': 'JSONDecodeError: Expecting value: line 1 column 1 (char 0)'}


,installation_name,installation_url,root_database_id,root_alias
0,Harvard_Dataverse,https://dataverse.harvard.edu,1.0,harvard
1,DataverseNL,https://dataverse.nl,1.0,root
2,Texas_Data_Repository_Dataverse,https://dataverse.tdl.org,NaN,None
3,DataverseNO,https://dataverse.no,1.0,root
4,Dataverse_e_cienciaDatos,https://edatos.consorciomadrono.es,1.0,Madrono
5,DANS_Data_Station_Social_Sciences_and_Humanities,https://ssh.datastations.nl,1.0,root
6,DANS_Data_Station_Life_Sciences,https://lifesciences.datastations.nl,1.0,root
7,DANS_Data_Station_Archaeology,https://archaeology.datastations.nl,1.0,root
8,BSC_Dataverse,https://dataverse.bsc.es,1.0,bsc
9,DANS_Data_Station_Physical_and_Technical_Sciences,https://phys-techsciences.datastations.nl,1.0,root


# Save the root collection aliases
Saves a new CSV file. No raw source data files are overwritten.

In [5]:
rootAliasesDf.to_csv(outputFilePath, index=False, encoding='utf-8-sig')
print(f'Saved {len(rootAliasesDf)} rows to {outputFilePath}')

Saved 14 rows to /Users/juliangautier/ux-research-data-analysis/Findability_of_author_affiliations_with_rors/data/processed/root_collection_aliases.csv
